In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib.pyplot as plt


In [ ]:
df_races = pd.read_csv("csv/races.csv")
df_laps = pd.read_csv("csv/laps.csv") 

In [ ]:
hilfe_map = {
    1: "No Assistance",
    2: "With Driving Coach",
    3: "With Ideal Driving Line"
}

df_races["help_label"] = df_races["help"].map(hilfe_map)
df_laps["help_label"] = df_laps["help"].map(hilfe_map)


In [ ]:
grouped = (
    df_races
    .groupby(["help", "racetrack"])["total_duration"]
    .mean()
    .reset_index()
)

grouped

In [ ]:
mean_total = (
    df_races
    .groupby("help_label")["total_duration"]
    .mean()
    .reset_index()
)

plt.figure()
plt.bar(mean_total["help_label"], mean_total["total_duration"])
plt.xlabel("Help condition")
plt.ylabel("Average total duration (s)")
plt.title("Total duration by help condition")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("pictures/total_duration_by_help_condition.png")
plt.close()

In [ ]:
mean_lap = (
    df_laps
    .groupby(["help_label", "lap"])["duration"]
    .mean()
    .reset_index()
)

plt.figure()

for hilfe in mean_lap["help_label"].unique():
    subset = mean_lap[mean_lap["help_label"] == hilfe]
    plt.plot(subset["lap"], subset["duration"], marker="o", label=hilfe)

plt.xlabel("Lap number")
plt.ylabel("Average lap duration (s)")
plt.title("Lap Duration by Assistance System")
plt.legend()
plt.tight_layout()
plt.savefig("pictures/lap_improvement_by_help_condition.png")
plt.close()

In [ ]:
df_races_encoded = pd.get_dummies(
    df_races,
    columns=["racetrack", "help", "experience", "race_number"],
    drop_first=True
)

df_laps_encoded = pd.get_dummies(
    df_laps,
    columns=["racetrack", "lap", "help", "experience", "race_number"],
    drop_first=True
)

df_races_encoded.columns = [
    "name",
    "id",
    "total_duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]


df_laps_encoded.columns = [
    "name",
    "id",
    "duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "lap_2",
    "lap_3",
    "lap_4",
    "lap_5",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]

df_races_encoded.to_csv("csv/races_oneHot.csv", index=False)
df_laps_encoded.to_csv("csv/laps_oneHot.csv", index=False) 

In [ ]:
X = df_races_encoded[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "experience_2",
            "experience_3",
            "race_number_2",
            "race_number_3",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
lap_cols = ["lap_2","lap_3","lap_4","lap_5"]

df_laps_lap1 = df_laps_encoded.loc[
    (df_laps_encoded["lap_2"] == 0) &
    (df_laps_encoded["lap_3"] == 0) &
    (df_laps_encoded["lap_4"] == 0) &
    (df_laps_encoded["lap_5"] == 0)
]

df_laps_lap1 = df_laps_encoded.loc[
    (df_laps_encoded[lap_cols] == 0).all(axis=1)
]

X = df_laps_lap1[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "experience_2",
            "experience_3",
            "race_number_2",
            "race_number_3",
        ]
    ].astype(float)
y = df_laps_lap1["duration"].astype(float)

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
from statsmodels.iolib.summary2 import summary_col

summary = summary_col(
    [model],
    stars=True,
    model_names=["OLS"],
    info_dict={
        "N":lambda x: f"{int(x.nobs)}",
        "R2":lambda x: f"{x.rsquared:.3f}"
    }
)

print(summary.as_latex())

In [ ]:
X = df_races_encoded[
        [
            "help_2",
            "help_3",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
X = df_races_encoded[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "race_number_2",
            "race_number_3",
            "three_race_duration"
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
X = df_laps_encoded[
        [
            "help_2",
            "help_3",
            "lap_2",
            "lap_3",
            "lap_4",
            "lap_5",
        ]
    ]
y = df_laps_encoded["duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)


formula = "duration ~ help_2 + help_3 + lap_2 + lap_3 + lap_4 + lap_5 + " \
    "lap_2*help_2 + lap_3*help_2 + lap_4*help_2 + lap_5*help_2 + " \
    "lap_2*help_3 + lap_3*help_3 + lap_4*help_3 + lap_5*help_3"
            
model = sm.OLS.from_formula(formula, data=df_laps_encoded).fit()

print(model.summary())

In [ ]:

lap_cols = [c for c in df_laps_encoded.columns if c.startswith("lap_")]
lap_cols = sorted(lap_cols, key=lambda s: int(s.split("_")[1]))

def _decode_lap(row):
    for col in lap_cols:
        if row[col] == 1:
            return int(col.split("_")[1])
    return 1

df_laps_encoded["lap"] = df_laps_encoded.apply(_decode_lap, axis=1)

df_laps_encoded.to_csv("csv/laps_oneHot_decoded.csv", index=False)

In [ ]:
X = df_laps_encoded[
        [
            "help_2",
            "help_3",
            "lap_2",
            "lap_3",
            "lap_4",
            "lap_5",
            "lap"
        ]
    ]
y = df_laps_encoded["duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

formula = "duration ~ help_2 + help_3 + lap + lap*help_2 + lap*help_3"
            
model = sm.OLS.from_formula(formula, data=df_laps_encoded).fit()

print(model.summary())
